# Stage 6 — Multi-Coin Robustness

Apply the **chosen** configuration (locked at Stage 3) to 5 alternative symbols over the full 4-year date range — no re-optimization per coin.

**Acceptance criterion:** ≥ 4 of 6 symbols (BTC + 5 alts) must pass the Stage 5 thresholds.

A strategy that only works on BTC is treated as overfit to BTC's idiosyncratic history.

In [ ]:
from __future__ import annotations

import json
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm

from quant.backtest.costs import CostMode, get_cost_model
from quant.backtest.metrics import compute_metrics
from quant.data.loader import load_ohlcv_parquet
from quant.strategies.mean_reversion import MeanReversionConfig, MeanReversionStrategy

DATA_RAW = Path('../data/raw')
DATA_PROC = Path('../data/processed')
RESULTS = Path('../results')
SYMBOLS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'AVAXUSDT', 'XRPUSDT', 'BNBUSDT']
INTERVAL = '1h'

PF_MIN = 1.3
SHARPE_MIN = 1.0
MAX_DD_MAX = 0.30

In [ ]:
chosen = json.loads((DATA_PROC / 'chosen_config.json').read_text())
params = chosen['params']

base = MeanReversionConfig(name='mean_reversion', timeframe='1h')
config = MeanReversionConfig(**{**asdict(base), **params})
cost = get_cost_model(CostMode.HORROR)
strat = MeanReversionStrategy(config)

print('Chosen config:')
print(json.dumps(params, indent=2))

In [ ]:
rows = []
for sym in tqdm(SYMBOLS):
    df = load_ohlcv_parquet(DATA_RAW / f'{sym}_{INTERVAL}.parquet')
    trades = strat.simulate(df, cost_model=cost)
    metrics = compute_metrics(trades)
    gates_passed = (
        metrics['pf'] > PF_MIN
        and metrics['sharpe'] > SHARPE_MIN
        and metrics['max_dd'] < MAX_DD_MAX
    )
    rows.append({
        'symbol': sym,
        'n_trades': metrics['n_trades'],
        'win_rate': metrics['win_rate'],
        'pf': metrics['pf'],
        'sharpe': metrics['sharpe'],
        'max_dd': metrics['max_dd'],
        'cagr': metrics['cagr'],
        'passes': gates_passed,
    })

multi = pd.DataFrame(rows)
multi.round(4)

In [ ]:
n_pass = int(multi['passes'].sum())
print(f'Symbols passing Stage 5 thresholds: {n_pass} / {len(multi)}')
print(f'Stage 6 verdict: {"PASS" if n_pass >= 4 else "FAIL — strategy fragile across symbols"}')

(RESULTS / 'tables').mkdir(parents=True, exist_ok=True)
multi.to_csv(RESULTS / 'tables' / 'multi_coin_results.csv', index=False)
print(f'Wrote: results/tables/multi_coin_results.csv')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, threshold, color in [
    (axes[0], 'pf',     PF_MIN,     '#3b82f6'),
    (axes[1], 'sharpe', SHARPE_MIN, '#10b981'),
    (axes[2], 'max_dd', MAX_DD_MAX, '#ef4444'),
]:
    bars = ax.bar(multi['symbol'], multi[col], color=color, alpha=0.7, edgecolor='black', lw=0.5)
    ax.axhline(threshold, color='black', linestyle='--', lw=1, alpha=0.6, label=f'threshold = {threshold}')
    ax.set_title(col.upper())
    ax.set_ylabel(col)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.setp(ax.get_xticklabels(), rotation=30)

plt.tight_layout()
out = RESULTS / 'figures' / '07_multicoin_metrics.png'
fig.savefig(out, dpi=120, bbox_inches='tight')
print(f'Wrote: {out}')
plt.show()